In [2]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from functions_v2 import *
from two_level_mc import *
from graph_mlmc_model import GraphTwoLevelModel
from mlmc_runner import MLMCRunner
from sksparse.cholmod import cholesky as sparse_cholesky

In [5]:
edges, n_vertices, edge_weights = load_graph(r"../../data/raw/power-US-Grid.mtx")
num_components = check_connectivity(edges, n_vertices)
if num_components > 1:
    edges, n_vertices, edge_weights = filter_to_largest_component(edges, n_vertices, edge_weights)

A, D, L = build_graph_matrices(edges, n_vertices)
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)

import networkx as nx
G_nx = nx.Graph()
G_nx.add_edges_from(edges)
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

✓ Loaded: ../../data/raw/power-US-Grid.mtx
  Vertices : 4941
  Edges    : 6594
  Weighted : no

Components: 1
  -> Graph is fully connected, safe to proceed

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 4941 x 4941
  Degree range: [1, 19]
  Non-zeros in L: 18129

✓ lambda_min = 0.000271
  (eigenvalues found: [0.         0.00027102])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse



In [6]:
setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

gamma_in: 23, gamma_out: 14, n_vertices: 4941
Boundary fraction: 0.7488%
  -> Likely safe for aggregation (comparable to validated successes).
n_coarse: 2056 (2056 aggregates)
Size distribution -- min: 1, max: 10, mean: 2.40
Singletons: 7 (0.3%)
gamma_in_coarse: 11, gamma_out_coarse: 4
Overlap (must be empty): set()
Interior coarse vertices: 2041 (99.3%)
coarse edges: 3260 (from 6594 fine edges)


In [13]:
model = GraphTwoLevelModel(setup)
runner = MLMCRunner(model, base_seed=0)
result = runner.run_fixed(samples_per_level=[2000, 300])

print(f"Estimate: {result.estimate:.6f}")
print(f"Standard error: {result.standard_error:.6f}")

for lr in result.level_results:
    print(f"level={lr.level}, n={lr.sample_count}, mean_correction={lr.mean_correction:.6f}, "
          f"mean_cost={lr.mean_sample_cost:.6f}")

Estimate: 27.909699
Standard error: 15.977836
level=0, n=2000, mean_correction=31.907528, mean_cost=0.031737
level=1, n=300, mean_correction=-3.997829, mean_cost=0.085450


In [15]:
# run on my independent code 
result_direct = run_paired_validation(setup, N=300)
print(f"Correlation: {result_direct['correlation']:.4f}")
print(f"Variance reduction: {result_direct['variance_reduction']:.2f}x")
print(f"Q_fine mean: {result_direct['Q_fine_samples'].mean():.6f}")
print(f"Q_coarse mean: {result_direct['Q_coarse_samples'].mean():.6f}")

estimate_direct = two_level_estimate(setup, result_direct, N_coarse_only=2000)
print(f"Direct two-level estimate: {estimate_direct['estimate']:.6f}")

Paired samples: 100%|██████████| 300/300 [00:26<00:00, 11.34sample/s, Q_fine=7.2139, Q_coarse=10.9644] 



N = 300 paired samples
Q_fine   : mean=7.213877  var=2385.803171
Q_coarse : mean=10.964447  var=5566.866596
Q_fine - Q_coarse : mean=-3.750570  var=664.056741
Correlation(Q_fine, Q_coarse): 1.0000
Variance reduction: 3.59x
Correlation: 1.0000
Variance reduction: 3.59x
Q_fine mean: 7.213877
Q_coarse mean: 10.964447


Coarse-only samples: 100%|██████████| 2000/2000 [01:33<00:00, 21.34sample/s, Q_coarse=11.5382]


Coarse-only base estimate (N=2000): 11.538190
Correction term mean (paired samples): -3.750570
Two-level estimate of E[Q_fine]: 7.787620
Direct fine-only mean (for comparison): 7.213877
Direct two-level estimate: 7.787620


In [29]:
single_level_se_powergrid = np.sqrt(2385.803171 / 300)
print(f"SE: {single_level_se_powergrid:.6f}")

SE: 2.820049


In [31]:
Q_coarse_only_samples = np.array([
    run_coarse_only_sample(setup, seed=100000+n) for n in range(2000)
])
print(f"Coarse-only mean: {Q_coarse_only_samples.mean():.6f}")
print(f"Coarse-only variance: {Q_coarse_only_samples.var():.6f}")

Coarse-only mean: 11.538190
Coarse-only variance: 9173.760837


In [33]:
se_correction = np.sqrt(664.056741 / 300)   # from your paired run's diff variance
se_coarse = np.sqrt(Q_coarse_only_samples.var() / 2000)
combined_se = np.sqrt(se_correction**2 + se_coarse**2)

print(f"SE from correction: {se_correction:.6f}")
print(f"SE from coarse-only: {se_coarse:.6f}")
print(f"Combined two-level SE: {combined_se:.6f}")

SE from correction: 1.487791
SE from coarse-only: 2.141700
Combined two-level SE: 2.607758
